In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import Lasso, ElasticNet, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.base import clone
from backend_aipw import *

In [ ]:
# DGP: sample size dominates number of groups
def dgp1(n, rng):
    p = 5
    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    beta0 = np.array([0.5, 1.0, 0.2, 0.4, 1.5])
    beta1 = beta0 + 1.0  # constant ATE = 1

    Y0 = X @ beta0 + rng.normal(0, 1, size=n)
    Y1 = X @ beta1 + rng.normal(0, 1, size=n)
    Y = D * Y1 + (1 - D) * Y0
    
    return X, D, Y, 2.5

In [ ]:
# DGP: number of groups relative to sample size converges to constant in [0, 1)
def dgp2(n, rng):
    p = int(0.1 * n)
    s = 5

    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    group_vals = rng.uniform(0, 1, size=s)
    beta = np.resize(group_vals, p)

    Y0 = X @ beta + rng.normal(0, 1, size=n)
    Y1 = Y0 + 1.0

    Y = D * Y1 + (1 - D) * Y0
    return X, D, Y, 1.0

In [ ]:
# DGP: number of groups relative to sample size converges to constant in [1, ∞) 
def dgp3(n, rng):
    p = int(1.7 * n)

    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)
 
    beta = rng.uniform(0, 1, size=p)

    Y0 = X @ beta + rng.normal(0, 1, size=n)
    Y1 = Y0 + 1.0

    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
# Simple Parameter Tuning
LASSO_GRID = {
    "alpha": np.logspace(-3, 0, 8),
}

EN_GRID = {
    "alpha": np.logspace(-3, 0, 8),
    "l1_ratio": [0.2, 0.5, 0.8],
}

RF_GRID = {
    "max_depth": [3, 5, None],
    "min_samples_leaf": [5, 10, 20],
}

GB_GRID = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3],
}

In [ ]:
# Tuning on small space
def tune_once(dgp_func, learners, n, seed=0):
    rng = np.random.default_rng(seed)
    X, D, Y, _ = dgp_func(n, rng)

    tuned = {}

    for name, model in learners.items():
        if name == "Lasso":
            grid = LASSO_GRID
        elif name == "ElasticNet":
            grid = EN_GRID
        elif name == "RF":
            grid = RF_GRID
        elif name == "GB":
            grid = GB_GRID
        else:
            tuned[name] = clone(model)
            continue

        gs = GridSearchCV(
            clone(model),
            grid,
            cv=3,
            scoring="neg_mean_squared_error",
            n_jobs=1
        )
        gs.fit(X, Y)
        tuned[name] = gs.best_estimator_

    return tuned

In [ ]:
learners_regime = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(max_iter=5000),
    "ElasticNet": ElasticNet(max_iter=5000),
    "RF": RandomForestRegressor(
        n_estimators=300,
        random_state=123,
        n_jobs=1
    ),
    "GB": GradientBoostingRegressor(
        n_estimators=300,
        random_state=123
    )
}

In [ ]:
# Non-Nested Sample Splitting
def run_simulation_fast(dgp_func, tuned_learners, n, seed):
    rng = np.random.default_rng(seed)
    X, D, Y, tau_true = dgp_func(n, rng)

    results = {}

    for name, learner in tuned_learners.items():
        m0_hat, m1_hat, e_hat = cross_fit_nuisances_fast(
            X, D, Y, learner, K=2
        )
        tau_hat, se, covered, _ = aipw(Y, D, m0_hat, m1_hat, e_hat, tau_true)
        results[name] = {
            "tau": tau_hat,
            "se": se,
            "covered": covered
        }
    return results

In [ ]:
# Non-Parallel Monte Carlo Simulation
def monte_carlo_serial(dgp_func, learners, n, sims=200):
    mc_results = Parallel(n_jobs=-1)(
        delayed(run_simulation_fast)(dgp_func, learners, n, s)
        for s in range(sims)
    )

    rows = []
    for sim in mc_results:
        for learner, res in sim.items():
            rows.append({
                "Learner": learner,
                "tau": res["tau"],
                "se": res["se"],
                "covered": res["covered"]
            })

    df = pd.DataFrame(rows)
    _, _, _, tau_true = dgp_func(n, np.random.default_rng(0))
    
    summary = df.groupby("Learner").agg(
        Mean=("tau", "mean"),
        Bias=("tau", lambda x: x.mean() - tau_true),
        Variance=("tau", "var"),
        RMSE=("tau", lambda x: np.sqrt(np.mean((x - tau_true) ** 2))),
        Coverage=("covered", "mean"),
        Mean_SE=("se", "mean")
    )

    return summary

In [ ]:
print("Regime 1")
tuned_learners_1 = tune_once(dgp1, learners_regime, n=100)
print(monte_carlo_serial(dgp1, tuned_learners_1, n=100, sims=100))
print("\n")

print("Regime 2")
tune_learners_2 = tune_once(dgp2, learners_regime, n=100)
print(monte_carlo_serial(dgp2, tune_learners_2, n=100, sims=100))
print("\n")

print("Regime 3")
tuned_learners_3 = tune_once(dgp3, learners_regime, n=100)
print(monte_carlo_serial(dgp3, tuned_learners_3, n=100, sims=100))

In [ ]:
# DGP Design Helpers
def make_group_design(n, rng, n_groups):
    """
    One-hot encoding of group instances.
    Each observation belongs to exactly one group.
    """
    group_id = rng.integers(0, n_groups, size=n)
    X = np.zeros((n, n_groups))
    X[np.arange(n), group_id] = 1.0
    return X, group_id

def make_group_sparse_beta(n_groups, s_unique, rng):
    """
    s_unique = number of unique coefficient values
    group sparsity = n_groups >> s_unique
    """
    unique_vals = rng.uniform(0.5, 1.5, size=s_unique)
    beta = np.resize(unique_vals, n_groups)
    return beta

In [ ]:
# Non-Sparse one hot encoded DGP
def dgp4(n, rng):
    n_groups = 6       # e.g. (Low/Mid/High) × (Male/Female)

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = rng.uniform(0.5, 1.5, size=n_groups)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
# Sparse one hot encoded DGP
def dgp5(n, rng):
    n_groups = int(0.1 * n)
    s_unique = 5       # strong group sparsity

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = make_group_sparse_beta(n_groups, s_unique, rng)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
# Anomalous DGP: number of groups exceeds sample size
"No latent factor"
def dgp_anomaly(n, rng):
    n_groups = int(2.5 * n)
    s_unique = n_groups  # no sparsity at all

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = rng.uniform(0.5, 1.5, size=n_groups)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
# DGP 4
print("DGP 4")
tuned_learners_4 = tune_once(dgp4, learners_regime, n=100)
print(monte_carlo_serial(dgp4, tuned_learners_4, n=100, sims=100))
print("\n")

# DGP 5
print("DGP 5")
tuned_learners_5 = tune_once(dgp5, learners_regime, n=100)
print(monte_carlo_serial(dgp5, tuned_learners_5, n=100, sims=100))
print("\n")

# DGP Anomaly
print("DGP Anomaly")
tuned_learners_anomaly = tune_once(dgp_anomaly, learners_regime, n=100)
print(monte_carlo_serial(dgp_anomaly, tuned_learners_anomaly, n=100, sims=100))